# Step 1 — Load first-level contrast maps

Load subject-level beta maps and build the design table used in second-level GLM.

**Prerequisites:** Edit `config.yaml` so paths point to your derivatives.

In [1]:
import sys
from pathlib import Path

import yaml

for _base in (Path.cwd(), *Path.cwd().parents[:6]):
    for _candidate in (_base, _base / "brain_activation"):
        if (_candidate / "config.yaml").is_file():
            ROOT = _candidate.resolve()
            break
    else:
        continue
    break
else:
    raise FileNotFoundError(
        "Could not find brain_activation/config.yaml. "
        "Open the brain_activation project in Jupyter, or cd to "
        "brain_activation/notebooks/ before starting the kernel."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PROJECT = ROOT.parent
RESULTS = ROOT / "results"
artifacts_dir = RESULTS / "artifacts"
res_path = RESULTS / "second_level"
cache_dir = RESULTS / "cachedir"
for _d in (artifacts_dir, res_path, cache_dir):
    _d.mkdir(parents=True, exist_ok=True)

def resolve_config_path(path_str):
    p = Path(path_str)
    return p.resolve() if p.is_absolute() else (ROOT / p).resolve()

cfg = yaml.safe_load((ROOT / "config.yaml").read_text(encoding="utf-8"))
PATHS = cfg["paths"]
ANALYSIS = cfg["analysis"]

nii_str = ANALYSIS["nii_str"]
first_level_path = resolve_config_path(PATHS["first_level_derivatives"]) / nii_str
mask_paths = resolve_config_path(PATHS["brain_mask"])

print("Package root:", ROOT)
print("Project root:", PROJECT)
print("Results dir:", RESULTS)
print("First-level path:", first_level_path)
print("Second-level output:", res_path)
print("Brain mask:", mask_paths)
if not first_level_path.exists():
    raise FileNotFoundError(
        f"First-level path not found: {first_level_path}\n"
        "Edit brain_activation/config.yaml"
    )


Package root: D:\文件同步\文件\数据分析结果\data_code_figure\brain_activation
Project root: D:\文件同步\文件\数据分析结果\data_code_figure
Results dir: D:\文件同步\文件\数据分析结果\data_code_figure\brain_activation\results
First-level path: D:\文件同步\文件\数据分析结果\data_code_figure\brain_activation\example_data\first_level\full_factorial
Second-level output: D:\文件同步\文件\数据分析结果\data_code_figure\brain_activation\results\second_level
Brain mask: D:\文件同步\文件\数据分析结果\data_code_figure\brain_activation\example_data\gray_mask.nii


In [2]:
import os.path as op
from glob import glob

import nibabel as nib

nii_files = []
subject_dirs = sorted(str(p) for p in first_level_path.glob("sub-*/patterns"))
if not subject_dirs:
    sample = sorted(first_level_path.glob("sub-*"))[:3]
    raise FileNotFoundError(
        f"No subject pattern folders found under {first_level_path}/sub-*/patterns\n"
        f"Resolved first-level path: {first_level_path}\n"
        f"Sample subject dirs: {[p.name for p in sample] or 'none'}"
    )

for subdir in subject_dirs:
    sub_nii_files = glob(op.join(subdir, "*-*_beta.nii.gz"))
    if len(sub_nii_files) > 1:
        sub_niis = [f for f in sub_nii_files if "error_trial" not in f]
    else:
        sub_niis = sub_nii_files
    nii_files.extend(sub_niis)

mask = nib.load(str(mask_paths))
print(f"Loaded mask: {mask_paths.name}")
print(f"Found {len(subject_dirs)} subjects and {len(nii_files)} first-level maps.")


Loaded mask: gray_mask.nii
Found 3 subjects and 24 first-level maps.


In [3]:
import re

import pandas as pd

factors = sorted(["long_short", "probability", "transition"])
data = []

for file_path in sorted(nii_files):
    file_name = op.basename(file_path)
    file_parts = re.split("[_-]", file_name)
    raw_subject = f"sub-{file_parts[1]}"
    file_data = {
        "subject": raw_subject,
        "probability": file_parts[4],
        "long_short": file_parts[2],
        "transition": file_parts[5],
    }
    data.append(file_data)

df = pd.DataFrame(data)
raw_subjects = sorted(df["subject"].unique())
sub_id_map = {raw: f"sub-{idx + 1:03d}" for idx, raw in enumerate(raw_subjects)}
df["subject"] = df["subject"].map(sub_id_map)

df_factor = df[factors].copy()
df_dum = df[["subject"]].copy()

print("Design factors:", factors)
print(f"N subjects: {df['subject'].nunique()}")
print(f"N first-level maps: {len(df)}")
df_factor.head()


Design factors: ['long_short', 'probability', 'transition']
N subjects: 3
N first-level maps: 24


,long_short,probability,transition
0,Long,diff,flexibility
1,Long,diff,stability
2,Long,same,flexibility
3,Long,same,stability
4,Short,diff,flexibility


In [4]:
df.to_csv(artifacts_dir / "design_df.csv", index=False)
df_factor.to_csv(artifacts_dir / "design_factors.csv", index=False)
df_dum.to_csv(artifacts_dir / "design_subject_dummies.csv", index=False)

print("Saved design tables to", artifacts_dir)
print(f"  subjects: {df['subject'].nunique()}, maps: {len(df)}")


Saved design tables to D:\文件同步\文件\数据分析结果\data_code_figure\brain_activation\results\artifacts
  subjects: 3, maps: 24
